
# Classificação otimizada de Auxílios em Andamento

Notebook oficial para preparar o dataframe simplificado, executar a classificação com o Gemini 2.5 Flash Lite e gerar os arquivos finais com rótulos, categorias e palavras-chave alinhados ao ICP da BioLinker.



## Fluxo
1. Configuração da API e dos parâmetros principais.
2. Normalização automática do CSV (detecta encoding/separador e garante colunas de interesse).
3. Execução em dois estágios com prompts especializados:
   - **Classificação (Etapa 1):** identifica se o projeto representa um ICP BioLinker e registra justificativa + palavras-chave.
   - **Extração (Etapa 2):** somente para clientes classificados, retornando códigos oficiais de categoria e palavras-chave contextuais adicionais.
4. Exportação dos resultados completos, incluindo os sinais utilizados na decisão do LLM.


In [ ]:
import os, json, math, time, random, unicodedata
from typing import Dict, List

import chardet
import pandas as pd
from google import genai
from google.genai import types
from IPython.display import display

# ======================================================================================
# Configurações principais
# ======================================================================================
GEMINI_API_KEY = os.getenv("GOOGLE_API_KEY") or ""
MODEL_NAME = "gemini-2.5-flash-lite"
INPUT_CSV_PATH = "/content/auxilios_em_andamento.csv"
OUTPUT_NAME = "classificacao_biolinker"
OUTPUT_CSV_PATH = f"/content/{OUTPUT_NAME}.csv"
BATCH_SIZE = 15
MAX_RETRIES = 3
SLEEP_BETWEEN_RETRIES = 5
TEMPERATURE = 0.0

if not GEMINI_API_KEY:
    raise RuntimeError("Defina GOOGLE_API_KEY ou preencha GEMINI_API_KEY manualmente.")

# ======================================================================================
# Contexto estratégico da BioLinker e ICP
# ======================================================================================
BIO_LINKER_CONTEXT = """
BioLinker: startup brasileira de biologia sintética sediada em Cotia (SP). Atua com engenharia genética, produção e purificação de proteínas recombinantes sob demanda, antígenos para diagnóstico/vacinas, kits educacionais e plataforma cell-free (CFPS) para proteínas tóxicas ou difíceis, com prototipagem muito rápida. Atende pesquisa acadêmica/industrial, diagnósticos, vacinas, cosméticos, agro e foodtech, oferecendo customização, escala e conformidade regulatória.
"""

ICP_KEYWORDS = {
    "proteinas_recombinantes": [
        "producao de proteina", "proteina recombinante", "purificacao de proteina", "antigeno recombinante",
        "proteina sob demanda", "expressao heterologa", "expressao de antigeno", "proteina terapeutica"
    ],
    "engenharia_genetica": [
        "engenharia genetica", "biologia sintetica", "sintese genica", "clonagem", "vetor de expressao",
        "gene editing", "crispr", "construcao genica", "circuito genetico"
    ],
    "ensaios_diagnosticos": [
        "elisa", "biossensor", "teste rapido", "imunoensaio", "diagnostico molecular", "lateral flow",
        "antigeno para teste", "validacao analitica"
    ],
    "plataforma_cell_free": [
        "cell-free", "sintese livre de celulas", "sistema acelular", "cfps", "prototipagem rapida",
        "proteina toxica", "proteina dificil", "hard-to-express"
    ],
    "bioquimica_analitica": [
        "proteomica", "cromatografia", "espectrometria de massas", "modelagem estrutural", "hplc",
        "purificacao escalar", "analise proteica"
    ],
    "mercados_aplicacao": [
        "diagnostico", "vacina", "cosmetico", "agro", "foodtech", "kit educacional", "bioprocesso",
        "biocatalise industrial"
    ],
}

CRITERIOS_KEYWORDS = {
    "PA1": [
        "expressao proteina", "purificacao proteina", "proteina recombinante", "antigeno recombinante",
        "expressao heterologa", "producao proteinas", "protein expression", "protein purification",
        "recombinant protein", "custom protein", "antigen production"
    ],
    "PA2": [
        "enzimas biotecnologicas", "caracterizacao enzimas", "purificacao enzimas", "biocatalise",
        "enzimas industriais", "enzyme characterization", "biocatalysis", "industrial enzyme", "enzyme panel"
    ],
    "PA3": [
        "elisa", "western blot", "biossensores", "imunoensaios", "triagem farmacos", "imunizacao",
        "biosensors", "drug screening", "diagnostico molecular", "lateral flow"
    ],
    "PA4": [
        "cromatografia", "espectrometria massas", "modelagem estrutural", "hplc", "proteomica",
        "chromatography", "mass spectrometry", "structural modeling", "protein analytics"
    ],
    "S1": [
        "sintese genica", "expressao genica", "construcao genica", "gene synthesis", "gene expression",
        "gene construction", "gene assembly"
    ],
    "S2": [
        "clonagem molecular", "pcr", "crispr", "edicao genetica", "molecular cloning", "gene editing",
        "genetic engineering", "vetor de expressao"
    ],
    "S3": [
        "circuito genetico", "chassis bacteriano", "engenharia metabolica", "biologia sintetica",
        "genetic circuits", "synthetic biology", "metabolic engineering", "cell-free design"
    ],
    "C1": [
        "cfps", "cell-free", "sintese livre celula", "sistema acelular", "cell-free protein synthesis",
        "in vitro protein synthesis", "cell-free kit", "cell-free prototyping"
    ],
    "C2": [
        "proteinas toxicas", "proteinas dificeis", "proteinas recalcitrantes", "toxic proteins",
        "difficult proteins", "hard-to-express proteins", "low yield"
    ],
    "C3": [
        "screening farmacos", "triagem medicamentos", "descoberta drogas", "drug discovery",
        "validacao expressao", "hit validation", "lead optimization"
    ],
    "C4": [
        "educacao", "ensino", "didatica", "educacional", "education", "teaching", "hands-on kit",
        "laboratorio didatico"
    ],
    "C5": [
        "cristalografia proteinas", "estrutura proteinas", "cristais proteina", "difracao raios x",
        "protein crystallography", "structural biology"
    ],
    "F1": [
        "cultura celular", "cultivo celulas", "diferenciacao celular", "celulas-tronco", "ipscs",
        "cell culture", "stem cells", "growth factors"
    ],
    "F2": [
        "fermentacao", "biorreatores", "crescimento celular", "producao biomassa", "fermentation",
        "bioreactors", "scaling up", "fed-batch"
    ],
    "F3": [
        "embriologia", "reproducao assistida", "fertilizacao in vitro", "desenvolvimento embrionario",
        "assisted reproduction", "cultivo embrionario"
    ],
    "F4": [
        "engenharia tecidos", "bioimpressao", "scaffolds", "medicina regenerativa", "tissue engineering",
        "bioprinting", "regenerative medicine"
    ],
}

# ======================================================================================
# Funções auxiliares
# ======================================================================================
def normalize(text: str) -> str:
    if text is None:
        return ""
    text = unicodedata.normalize("NFD", str(text))
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    return text.lower().strip()

def detect_encoding_and_sep(path: str):
    with open(path, "rb") as f:
        raw = f.read(200000)
    enc = chardet.detect(raw).get("encoding") or "utf-8"
    with open(path, "r", encoding=enc, errors="ignore") as f:
        head = f.readline()
    sep = ";" if head.count(";") >= head.count(",") else ","
    return enc, sep

def ensure_columns(df: pd.DataFrame) -> pd.DataFrame:
    colmap = {
        "Beneficiário": ["Beneficiário", "Beneficiario", "beneficiario", "Beneficiário "],
        "Instituição": ["Instituição", "Instituicao", "instituicao", "Instituição "],
        "Resumo (Português)": ["Resumo (Português)", "Resumo (Portugues)", "Resumo", "Resumo  (Português)"],
    }
    found = {}
    for target, options in colmap.items():
        for c in df.columns:
            if c.strip() in options:
                found[target] = c
                break
    missing = [k for k in colmap.keys() if k not in found]
    if missing:
        raise ValueError(f"Colunas ausentes no CSV: {missing}. Cabeçalho encontrado: {list(df.columns)}")
    return df.rename(columns={
        found["Beneficiário"]: "Beneficiário",
        found["Instituição"]: "Instituição",
        found["Resumo (Português)"]: "Resumo (Português)",
    })

def parse_json_strict(text: str):
    text = text.strip()
    if not (text.startswith("[") and text.endswith("]")):
        lb = text.find("[")
        rb = text.rfind("]")
        if lb != -1 and rb != -1 and rb > lb:
            text = text[lb : rb + 1]
    return json.loads(text)

def build_user_message(batch_items: List[Dict], etapa: str) -> str:
    return json.dumps(
        {
            "contexto_biolinker": BIO_LINKER_CONTEXT,
            "lote": batch_items,
            "etapa": etapa,
            "icp_keywords": ICP_KEYWORDS,
            "categorias_oficiais": CRITERIOS_KEYWORDS,
        },
        ensure_ascii=False,
    )

def call_gemini(user_msg: str, system_instruction: str, batch_number: int):
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        response_mime_type="application/json",
        temperature=TEMPERATURE,
    )
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=[types.Content(role="user", parts=[types.Part.from_text(text=user_msg)])],
                config=config,
            )
            data = parse_json_strict(response.text)
            if not isinstance(data, list):
                raise ValueError("Resposta não é uma lista JSON.")
            return data
        except Exception as exc:
            if attempt == MAX_RETRIES:
                print(f"Lote {batch_number}: falhou após {MAX_RETRIES} tentativas.")
                raise exc
            wait = (SLEEP_BETWEEN_RETRIES * (2 ** (attempt - 1))) + random.uniform(0, 1)
            print(f"Lote {batch_number}: tentativa {attempt}/{MAX_RETRIES} falhou ({exc}). Retentando em {wait:.1f}s...")
            time.sleep(wait)

# ======================================================================================
# Instruções avançadas para o Gemini
# ======================================================================================
SYSTEM_INSTRUCTION_CLASSIFY = f"""
Você é um analista técnico da BioLinker. Avalie resumos de projetos financiados e decida se representam ICP da BioLinker.

Contexto da empresa: {BIO_LINKER_CONTEXT}
Mapa de sinais ICP (para alinhamento mercadológico): {json.dumps(ICP_KEYWORDS, ensure_ascii=False)}
Dicionário de categorias oficiais (para mapeamento técnico): {json.dumps(CRITERIOS_KEYWORDS, ensure_ascii=False)}

Regras de decisão:
1. Leia cada resumo e identifique se há relação clara com síntese/engenharia de genes, expressão/purificação de proteínas, antígenos e ensaios diagnósticos, CFPS/cell-free, biologia molecular/celular, fatores de crescimento ou aplicações industriais/educacionais que demandem proteínas ou sistemas de expressão.
2. Se houver qualquer relação direta ou indício forte com os sinais acima, marque "cliente": true. Se o resumo não tiver conexão técnica, marque "cliente": false.
3. Justificativa: descreva em até 50 palavras a evidência principal (sem alucinar). Cite técnica, aplicação ou material do resumo.
4. palavras_chave_icp: forneça 3-8 termos separados por ponto (.) priorizando combinações do dicionário ICP e termos explícitos do resumo.
5. sinais_acionados: liste quais buckets do ICP ou códigos de categoria foram disparados (ex.: "proteinas_recombinantes", "plataforma_cell_free", "PA3"). Se nenhum, retorne lista vazia.
6. confianca: "alta", "media" ou "baixa" conforme clareza da evidência.
7. Formato obrigatório (lista JSON): [ {{"id": <int>, "cliente": <true|false>, "justificativa": "...", "palavras_chave_icp": "t1.t2", "sinais_acionados": ["..."], "confianca": "alta|media|baixa"}}, ... ]
8. Não invente fatos; use apenas o texto fornecido.
"""

SYSTEM_INSTRUCTION_EXTRACT = f"""
Você recebe apenas resumos já aprovados como clientes potenciais. Mapeie o enquadramento técnico.

Contexto de referência: {BIO_LINKER_CONTEXT}
Dicionário oficial de categorias: {json.dumps(CRITERIOS_KEYWORDS, ensure_ascii=False)}

Formato obrigatório (lista JSON): [ {{"id": <int>, "categorias": ["PA1", "C3"], "keywords": "termo1.termo2"}}, ... ]
Regra: retorne apenas códigos que apareçam no dicionário; se nenhuma categoria for clara, devolva lista vazia mas ainda forneça keywords técnicas (3-6 termos, separados por ponto).
"""

# ======================================================================================
# Leitura do CSV e preparação do dataframe simplificado
# ======================================================================================
client = genai.Client(api_key=GEMINI_API_KEY)

if not os.path.exists(INPUT_CSV_PATH):
    raise FileNotFoundError(f"CSV não encontrado em {INPUT_CSV_PATH}")

encoding, separator = detect_encoding_and_sep(INPUT_CSV_PATH)
df_raw = pd.read_csv(INPUT_CSV_PATH, encoding=encoding, sep=separator, engine="python")
df_raw = ensure_columns(df_raw)
df_raw["Beneficiário"] = df_raw["Beneficiário"].fillna("").astype(str)
df_raw["Instituição"] = df_raw["Instituição"].fillna("").astype(str)
df_raw["Resumo (Português)"] = df_raw["Resumo (Português)"].fillna("").astype(str)

# dataframe simplificado
df = df_raw.copy()
df["_tmp_id"] = range(len(df))
df["Cliente (valor booleano)"] = False
df["Justificativa LLM"] = ""
df["Palavras-chave ICP"] = ""
df["Sinais acionados"] = ""
df["Grau de confiança"] = ""
df['Categoria (strings separadas por ponto ".")'] = ""
df["Palavras-chave Categoria"] = ""

n_rows = len(df)
n_batches_e1 = math.ceil(n_rows / BATCH_SIZE)
print(f"Etapa 1: {n_rows} registros em {n_batches_e1} lote(s) de {BATCH_SIZE}.")

classification_results: Dict[int, bool] = {}
classification_reasons: Dict[int, str] = {}
classification_keywords: Dict[int, str] = {}
classification_signals: Dict[int, str] = {}
classification_confidence: Dict[int, str] = {}

for b in range(n_batches_e1):
    start = b * BATCH_SIZE
    end = min((b + 1) * BATCH_SIZE, n_rows)
    batch_df = df.iloc[start:end]
    batch_items = [
        {
            "id": int(row["_tmp_id"]),
            "beneficiario": row["Beneficiário"],
            "instituicao": row["Instituição"],
            "resumo_pt": row["Resumo (Português)"],
        }
        for _, row in batch_df.iterrows()
    ]
    user_msg = build_user_message(batch_items, etapa="classificacao")
    batch_out = call_gemini(user_msg, SYSTEM_INSTRUCTION_CLASSIFY, batch_number=b + 1)
    for item in batch_out:
        rid = int(item.get("id"))
        classification_results[rid] = bool(item.get("cliente", False))
        classification_reasons[rid] = str(item.get("justificativa", "")).strip()
        keywords = item.get("palavras_chave_icp") or item.get("palavras_chave") or ""
        classification_keywords[rid] = str(keywords).strip().strip(".")
        signals = item.get("sinais_acionados") or []
        if isinstance(signals, list):
            signals = ".".join(s.strip() for s in signals if s)
        classification_signals[rid] = str(signals)
        classification_confidence[rid] = str(item.get("confianca", "")).strip()

# Atualiza dataframe com resultados da Etapa 1
df["Cliente (valor booleano)"] = df["_tmp_id"].map(classification_results).fillna(False)
df["Justificativa LLM"] = df["_tmp_id"].map(classification_reasons).fillna("")
df["Palavras-chave ICP"] = df["_tmp_id"].map(classification_keywords).fillna("")
df["Sinais acionados"] = df["_tmp_id"].map(classification_signals).fillna("")
df["Grau de confiança"] = df["_tmp_id"].map(classification_confidence).fillna("")

# ======================================================================================
# Etapa 2: extração de categorias para clientes
# ======================================================================================
df_clients = df[df["Cliente (valor booleano)"] == True].copy()
num_clients = len(df_clients)
if num_clients > 0:
    n_batches_e2 = math.ceil(num_clients / BATCH_SIZE)
    extraction_results: Dict[int, str] = {}
    extraction_keywords: Dict[int, str] = {}
    print(f"Etapa 2: {num_clients} registros em {n_batches_e2} lote(s) de {BATCH_SIZE}.")
    for b in range(n_batches_e2):
        start = b * BATCH_SIZE
        end = min((b + 1) * BATCH_SIZE, num_clients)
        batch_df = df_clients.iloc[start:end]
        batch_items = [
            {"id": int(row["_tmp_id"]), "resumo_pt": row["Resumo (Português)"]}
            for _, row in batch_df.iterrows()
        ]
        user_msg = build_user_message(batch_items, etapa="extracao_categorias")
        batch_out = call_gemini(user_msg, SYSTEM_INSTRUCTION_EXTRACT, batch_number=b + 1)
        for item in batch_out:
            rid = int(item.get("id", -1))
            categorias = item.get("categorias") or []
            keywords = str(item.get("keywords", "")).strip().strip(".")
            categoria_str = ".".join(c.strip() for c in categorias if c.strip())
            extraction_results[rid] = categoria_str
            extraction_keywords[rid] = keywords
    df['Categoria (strings separadas por ponto ".")'] = df["_tmp_id"].map(extraction_results).fillna("")
    df["Palavras-chave Categoria"] = df["_tmp_id"].map(extraction_keywords).fillna("")
else:
    print("Nenhum cliente encontrado na Etapa 1. Etapa 2 ignorada.")

# ======================================================================================
# Exportação
# ======================================================================================
df_out = df.drop(columns=["_tmp_id"])
df_out = df_out[[
    "Beneficiário",
    "Instituição",
    "Resumo (Português)",
    "Cliente (valor booleano)",
    "Justificativa LLM",
    "Palavras-chave ICP",
    "Sinais acionados",
    "Grau de confiança",
    'Categoria (strings separadas por ponto ".")',
    "Palavras-chave Categoria",
]]

df_out.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8")
print(f"Arquivo final salvo em: {OUTPUT_CSV_PATH}")
print(f"Total de clientes potenciais: {int(df_out['Cliente (valor booleano)'].sum())} de {len(df_out)} registros.")

display(df_out.head(10))


